# MPB LaTeX OCR - Kaggle Dataset Training

Use this notebook when you want Kaggle's hosted datasets under `/kaggle/input` and a Kaggle GPU runtime. Attach a formula OCR dataset from the right sidebar before running the preparation cell.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=cwd)

run("nvidia-smi || true")
print("Attached Kaggle datasets:")
run("find /kaggle/input -maxdepth 2 -type d | sort | head -80")

## Get The Project

Set `REPO_URL` to your repository URL. If you uploaded the repo as a Kaggle dataset instead, copy it into `/kaggle/working/mpb-latex-ocr` before installing.

In [ ]:
REPO_URL = "https://github.com/Ackrome/mpb-latex-ocr.git"
PROJECT_DIR = Path("/kaggle/working/mpb-latex-ocr")

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL or copy the repo to /kaggle/working/mpb-latex-ocr first.")
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

SRC_DIR = PROJECT_DIR / "src"
os.environ["PYTHONPATH"] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"
print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)


## Install

Kaggle normally ships with a CUDA-enabled PyTorch build. This installs the local package and any missing dependencies.

In [ ]:
run("python -m pip install --upgrade pip", cwd=PROJECT_DIR)
run('python -m pip install --force-reinstall --no-deps -e ".[dev]"', cwd=PROJECT_DIR)
run("python -m pip install lightning mlflow omegaconf pillow matplotlib numpy tqdm pytest", cwd=PROJECT_DIR)
run("python scripts/kaggle_preflight.py", cwd=PROJECT_DIR)

import sys
sys.path.insert(0, str(SRC_DIR))

import mpb_latex_ocr
import mpb_latex_ocr.data
import mpb_latex_ocr.prepare_manifest
import torch
print("package:", mpb_latex_ocr.__file__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Choose Dataset Root

Point this at the attached dataset folder. Examples:

- `/kaggle/input/im2latex-230k/PRINTED_TEX_230k`
- `/kaggle/input/latex-formula-images`

The manifest converter supports common paired files such as `corresponding_png_images.txt` + `final_png_formulas.txt`, generic CSV/TSV/JSON/JSONL metadata, and some Im2LaTeX `.lst` split layouts.

In [ ]:
KAGGLE_DATASET_ROOT = Path("/kaggle/input/CHANGE_ME")
MANIFEST_PATH = Path("/kaggle/working/latex-ocr-manifest.csv")

if not KAGGLE_DATASET_ROOT.exists():
    raise FileNotFoundError(f"Set KAGGLE_DATASET_ROOT to an attached dataset folder: {KAGGLE_DATASET_ROOT}")

prepare_cmd = " ".join([
    "python -m mpb_latex_ocr.prepare_manifest",
    f"--input-root {KAGGLE_DATASET_ROOT}",
    f"--output {MANIFEST_PATH}",
    "--format auto",
    "--absolute-paths",
    "--max-samples 50000",
    "--val-fraction 0.05",
    "--test-fraction 0.05",
])
run(prepare_cmd, cwd=PROJECT_DIR)
run(f"head -5 {MANIFEST_PATH}")

## Train

Start with a capped subset and a few epochs. Increase `--max-samples`, epochs, and batch size only after the first run produces sensible predictions.

In [ ]:
train_cmd = " ".join([
    "python -m mpb_latex_ocr.train",
    "--config configs/train.yaml",
    "--config configs/hardware/kaggle.yaml",
    "--config configs/datasets/kaggle_manifest.yaml",
    "trainer.max_epochs=5",
    f"paths.manifest={MANIFEST_PATH}",
])
run(train_cmd, cwd=PROJECT_DIR)

## Evaluate

This computes string metrics, the fast render-aware proxy metrics, and exports `cdm_predictions.json` for official CDM tooling. Render metrics are slower, but they are much more informative than edit distance for LaTeX OCR.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/latex-ocr-runs/baseline")
checkpoints = sorted((OUTPUT_DIR / "checkpoints").glob("*.ckpt"))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found under {OUTPUT_DIR / 'checkpoints'}")
checkpoint = checkpoints[-1]
predictions_path = OUTPUT_DIR / "test_predictions.jsonl"
cdm_json_path = OUTPUT_DIR / "cdm_predictions.json"
print("checkpoint:", checkpoint)

eval_cmd = " ".join([
    "python -m mpb_latex_ocr.evaluate",
    f"--checkpoint {checkpoint}",
    f"--manifest {MANIFEST_PATH}",
    f"--tokenizer {OUTPUT_DIR / 'tokenizer.json'}",
    "--split test",
    "--render-metric",
    f"--predictions-out {predictions_path}",
    f"--cdm-json-out {cdm_json_path}",
])
run(eval_cmd, cwd=PROJECT_DIR)
print("predictions:", predictions_path)
print("cdm json:", cdm_json_path)
run(f"head -3 {predictions_path}")


## Visual Check On Im2LaTeX Samples

Run this after the evaluation cell. It displays actual dataset crops from `PRINTED_TEX_230k` alongside the target, prediction, and render-proxy score. This is the fastest way to see whether the checkpoint behaves better on real Kaggle images than on the local toy renderer.

In [ ]:
import json
import random
import textwrap

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from mpb_latex_ocr.metrics.render import render_formula_image

VISUAL_SAMPLE_COUNT = 8
VISUAL_SEED = 42
# Options: random, worst_render, best_render, first
VISUAL_SAMPLE_MODE = "random"

if not predictions_path.exists():
    raise FileNotFoundError(f"Run the evaluation cell first; missing {predictions_path}")

prediction_rows = [
    json.loads(line)
    for line in predictions_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if not prediction_rows:
    raise ValueError(f"No rows found in {predictions_path}")

def safe_formula_image(formula):
    try:
        return render_formula_image(formula)
    except Exception as exc:
        image = Image.new("L", (512, 128), color=255)
        draw = ImageDraw.Draw(image)
        draw.text((10, 10), f"render failed: {type(exc).__name__}\n{str(formula)[:120]}", fill=0)
        return image

def clipped_text(value, max_chars=360):
    value = str(value)
    if len(value) > max_chars:
        value = value[:max_chars] + " ..."
    return "\n".join(textwrap.wrap(value, width=72, break_long_words=True))

rows = list(prediction_rows)
if VISUAL_SAMPLE_MODE == "random":
    random.Random(VISUAL_SEED).shuffle(rows)
elif VISUAL_SAMPLE_MODE == "worst_render":
    rows.sort(key=lambda row: float(row.get("render_f1", 0.0)))
elif VISUAL_SAMPLE_MODE == "best_render":
    rows.sort(key=lambda row: float(row.get("render_f1", 0.0)), reverse=True)
elif VISUAL_SAMPLE_MODE != "first":
    raise ValueError(f"Unknown VISUAL_SAMPLE_MODE: {VISUAL_SAMPLE_MODE}")

sample_rows = rows[: min(VISUAL_SAMPLE_COUNT, len(rows))]
summary_keys = ["exact_match", "norm_edit_distance", "render_f1", "prediction_rendered", "target_rendered"]
summary = {"num_rows": len(prediction_rows)}
for key in summary_keys:
    if key in prediction_rows[0]:
        summary[key] = sum(float(row.get(key, 0.0)) for row in prediction_rows) / len(prediction_rows)
print(json.dumps(summary, indent=2))

fig, axes = plt.subplots(
    len(sample_rows),
    4,
    figsize=(20, 3.2 * max(1, len(sample_rows))),
    gridspec_kw={"width_ratios": [1.2, 1.2, 1.2, 2.6]},
)
if len(sample_rows) == 1:
    axes = [axes]

for row_axes, row in zip(axes, sample_rows, strict=True):
    input_image = Image.open(row["image_path"]).convert("L")
    target_image = safe_formula_image(row["target"])
    prediction_image = safe_formula_image(row["prediction"])

    for axis, image, title in [
        (row_axes[0], input_image, "input crop"),
        (row_axes[1], target_image, "target render"),
        (row_axes[2], prediction_image, f"prediction render | f1={float(row.get('render_f1', 0.0)):.3f}"),
    ]:
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_title(title)
        axis.axis("off")

    row_axes[3].axis("off")
    row_axes[3].set_title("raw strings")
    row_axes[3].text(
        0,
        1,
        "image: " + str(row.get("image_path", "")) + "\n\n"
        + "target:\n" + clipped_text(row.get("target", ""), 180) + "\n\n"
        + "prediction:\n" + clipped_text(row.get("prediction", "")),
        va="top",
        ha="left",
        family="monospace",
        fontsize=8,
    )

plt.tight_layout()
plt.show()


## Export Portable Im2LaTeX Sample Bundle

Run this after evaluation to copy exact image files referenced by `test_predictions.jsonl` into a portable bundle. Download the resulting zip and extract it locally to `data/im2latex_sample_bundle` for local visual checks and local checkpoint inference.

In [ ]:
EXPORT_SAMPLE_COUNT = 96
EXPORT_SAMPLE_MODE = "random"  # first, random, best-render, worst-render, prediction-render-fail, render-fail, exact-mismatch
EXPORT_SAMPLE_SEED = 42
SAMPLE_BUNDLE_DIR = OUTPUT_DIR / "im2latex_sample_bundle"
SAMPLE_BUNDLE_ZIP = OUTPUT_DIR / "im2latex_sample_bundle.zip"

if not predictions_path.exists():
    raise FileNotFoundError(f"Run the evaluation cell first; missing {predictions_path}")

export_cmd = " ".join([
    "python -m mpb_latex_ocr.export_prediction_samples",
    f"--predictions {predictions_path}",
    f"--output-dir {SAMPLE_BUNDLE_DIR}",
    f"--num-samples {EXPORT_SAMPLE_COUNT}",
    f"--mode {EXPORT_SAMPLE_MODE}",
    f"--seed {EXPORT_SAMPLE_SEED}",
    f"--zip-out {SAMPLE_BUNDLE_ZIP}",
])
run(export_cmd, cwd=PROJECT_DIR)
print("bundle dir:", SAMPLE_BUNDLE_DIR)
print("bundle zip:", SAMPLE_BUNDLE_ZIP)
print("download this zip, then extract locally to data/im2latex_sample_bundle")


## Save Outputs

Kaggle keeps `/kaggle/working` outputs as notebook artifacts when you save a version. Checkpoints, MLflow runs, the tokenizer, predictions, render-aware per-sample metrics, and `cdm_predictions.json` are under `/kaggle/working`.


## Optional: Official CDM

Run this after the evaluation cell has created `cdm_predictions.json`. Official CDM is slower and has heavier system dependencies than the in-repo render proxy. Kaggle notebooks cannot run Docker, so this cell uses a direct Linux install path.

Keep this disabled until you need final CDM numbers. If dependency installation fails or takes too long, download `cdm_predictions.json` and run the Docker workflow locally.


In [ ]:
RUN_OFFICIAL_CDM = False
CDM_POOLS = 4
CDM_REPO_DIR = Path("/kaggle/working/UniMERNet")
CDM_DIR = CDM_REPO_DIR / "cdm"
CDM_JSON_PATH = Path("/kaggle/working/latex-ocr-runs/baseline/cdm_predictions.json")
CDM_OUTPUT_DIR = Path("/kaggle/working/latex-ocr-runs/baseline/official_cdm")

if RUN_OFFICIAL_CDM:
    if not CDM_JSON_PATH.exists():
        raise FileNotFoundError(f"Run the evaluation cell first; missing {CDM_JSON_PATH}")

    # Kaggle cannot run Docker in notebooks, so install the runtime dependencies directly.
    # This is intentionally lighter than texlive-full; if CDM rendering fails on your formulas,
    # replace these TeX packages with texlive-full and rerun the cell.
    run("sudo apt-get update")
    run("sudo apt-get install -y nodejs npm imagemagick poppler-utils texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-science cm-super")
    run("sudo sed -i 's/rights=\"none\" pattern=\"PDF\"/rights=\"read|write\" pattern=\"PDF\"/' /etc/ImageMagick-6/policy.xml || true")

    if not CDM_DIR.exists():
        run(f"git clone --depth 1 https://github.com/opendatalab/UniMERNet.git {CDM_REPO_DIR}")

    # The official requirements pin old versions; on Kaggle/Python 3.12, fall back to current scikit-image if needed.
    run("python -m pip install \"numpy<2.0.0\" tqdm matplotlib opencv-python", cwd=CDM_DIR)
    run("python -m pip install \"scikit-image<=0.20.0\" || python -m pip install scikit-image", cwd=CDM_DIR)

    CDM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cdm_cmd = " ".join([
        "python evaluation.py",
        f"-i {CDM_JSON_PATH}",
        f"-o {CDM_OUTPUT_DIR}",
        f"-p {CDM_POOLS}",
    ])
    run(cdm_cmd, cwd=CDM_DIR)
    metrics_path = CDM_OUTPUT_DIR / CDM_JSON_PATH.stem / "metrics_res.json"
    print("official CDM metrics:", metrics_path)
    run(f"cat {metrics_path}")
else:
    print("Official CDM is disabled. Set RUN_OFFICIAL_CDM = True after cdm_predictions.json exists.")
